In [1]:
from fastembed import TextEmbedding
from sentence_transformers import SentenceTransformer
import numpy as np
from qdrant_client import QdrantClient, models
import csv
import json
import numpy as np
import enum
import pandas as pd

In [2]:
def load_ground_truth():
    records=[]
    with open('ground-truth-data.csv', 'r') as file:
        csv_reader = csv.reader(file)
        for row in csv_reader:
            records.append(row)
        return records
    
def load_documents():
    documents = []
    with open('documents-with-ids.json', 'rb') as f_in:
        documents = json.load(f_in)
    return documents

def filter_ground_truth(filter:str):
    records = load_ground_truth()
    df_ground_truth = pd.DataFrame(records)    
    df_ground_truth = df_ground_truth[df_ground_truth[1] == filter]
    ground_truth = df_ground_truth.to_dict(orient='records')
    return ground_truth

In [3]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [4]:
model_name = "jinaai/jina-embeddings-v2-small-en"
EMBEDDING_DIMENSIONALITY = 512

In [5]:
client=None

In [12]:
def setup_minsearch_qdrant(distance):
    global client
    client = QdrantClient("http://localhost:6333/")
    
    model = TextEmbedding(model_name=model_name)
    # Define the collection name
    collection_name = "zoomcamp-rag"

    # Create the collection with specified vector parameters
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=EMBEDDING_DIMENSIONALITY,  # Dimensionality of the vectors
            distance=distance  # Distance metric for similarity search
        )
    )

    documents=load_documents()
    points = []
    id = 0

    for doc in documents[:100]:
        question = doc['question']
        text = doc['text']
        course=doc['course']
        section=doc['section']
        doc_id=doc['id']
        qnText = doc['question'] + ' ' + doc['text']
        point = models.PointStruct(
            id=id,
            vector=models.Document(text=qnText, model=model_name), #embed text locally with "jinaai/jina-embeddings-v2-small-en" from FastEmbed
            payload={
                "question": question,
                "text": text,
                "section": section,
                "course": course,
                "doc_id":doc_id
            } #save all needed metadata fields
        )
        points.append(point)
        id += 1

        client.upsert(
            collection_name=collection_name,
            points=points
        )

In [13]:
distance=models.Distance.COSINE
setup_minsearch_qdrant(distance)

In [14]:
def minsearch_vector_search_qdrant(question,course):
    collection_name = "zoomcamp-rag"
    results = client.query_points(
        collection_name=collection_name,
        query=models.Document( #embed the query text locally with "jinaai/jina-embeddings-v2-small-en"
            text=question,
            model=model_name 
        ),
        limit=5, # top closest matches
        with_payload=True #to get metadata in the results
    )
    return results

In [15]:
def compute_relevance(record, result):
    doc_id=record[2]
    doc_ids = [point.payload['doc_id'] for point in result.points]
    print(doc_id)
    print(doc_ids)
    relevance=[id==doc_id for id in doc_ids]
    print(relevance)
    return relevance

## Computing relevances

# Computing results
def compute_results(ground_truth):
    results=[]
    relevances=[]
    
    for record in ground_truth[:75]:
        question=record[0]
        course=record[1]
        result=minsearch_vector_search_qdrant(question,course)
        relevance=compute_relevance(record,result)
        results.append(result)
        relevances.append(relevance)
    return results,relevances

In [16]:
filter:str='machine-learning-zoomcamp'
ground_truth=filter_ground_truth(filter)
results, relevances = compute_results(ground_truth)
print("hit_rate:" + str(hit_rate(relevances)))
print("mrr:" + str(mrr(relevances)))    

0227b872
['0bbf41ec', 'a482086d', 'c02e79ef', '7842b56a', 'cb257ee5']
[False, False, False, False, False]
0227b872
['0bbf41ec', '543ff080', '29865466', '016d46a1', 'c02e79ef']
[False, False, False, False, False]
0227b872
['0bbf41ec', '251218fc', 'c02e79ef', 'f2945cd2', 'e866156b']
[False, False, False, False, False]
0227b872
['a482086d', '29865466', 'c02e79ef', '2f19301f', 'eb56ae98']
[False, False, False, False, False]
0227b872
['1f6520ca', '29865466', 'c02e79ef', '0bbf41ec', '0e424a44']
[False, False, False, False, False]
39fda9f0
['c02e79ef', 'a482086d', '9681be3b', 'cb257ee5', '7842b56a']
[False, False, False, False, False]
39fda9f0
['c02e79ef', 'a482086d', '7842b56a', 'ea739c65', 'cb257ee5']
[False, False, False, False, False]
39fda9f0
['9681be3b', '52393fb3', '04aa4897', 'c02e79ef', '2ed9b986']
[False, False, False, False, False]
39fda9f0
['9681be3b', '04aa4897', '52393fb3', 'c02e79ef', 'be5bfee4']
[False, False, False, False, False]
39fda9f0
['04aa4897', 'c02e79ef', '9681be3b', 

In [29]:
from sentence_transformers import SentenceTransformer

model_name = 'multi-qa-MiniLM-L6-cos-v1'
model = SentenceTransformer(model_name)

In [19]:
def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)

In [36]:
def load_cosine_results():
    cosine_results=[]
    with open('results-gpt4o-mini.csv', 'r') as file:
        csv_reader = csv.reader(file)
        for row in csv_reader:
            cosine_results.append(row)
        return cosine_results
    
df_results=pd.DataFrame()

def return_cosine_results_df():
    global df_results
    records = load_cosine_results()
    df_results = pd.DataFrame(records)    
    rag_real_results = df_results.to_dict(orient='records')
    return rag_real_results

In [37]:
rag_real_results=return_cosine_results_df()

In [30]:
def compute_similarity(record):
    answer_orig = record[2]
    answer_llm = record[1]
    
    v_llm = model.encode(answer_llm)
    v_orig = model.encode(answer_orig)
    
    return cosine(v_orig,v_llm)

In [32]:
similarity = []

for record in rag_real_results:
    sim = compute_similarity(record)
    print(sim)
    similarity.append(sim)
    

0.017293748
0.07906622
0.07906622
0.07906622
0.07906622
0.07906622
-0.078001134
-0.078001134
-0.078001134
-0.078001134
-0.078001134
0.034810912
0.034810912
0.034810912
0.034810912
0.034810912
-0.03447833
-0.03447833
-0.03447833
-0.03447833
-0.03447833
0.06504624
0.06504624
0.06504624
0.06504624
0.06504624
0.13638534
0.13638534
0.13638534
0.13638534
0.13638534
-0.054196224
-0.054196224
-0.054196224
-0.054196224
-0.054196224
-0.06447325
-0.06447325
-0.06447325
-0.06447325
-0.06447325
-0.022965401
-0.022965401
-0.022965401
-0.022965401
-0.022965401
-0.023871252
-0.023871252
-0.023871252
-0.023871252
-0.023871252
0.021369366
0.021369366
0.021369366
0.021369366
0.021369366
-0.14982645
-0.14982645
-0.14982645
-0.14982645
-0.14982645
0.13373758
0.13373758
0.13373758
0.13373758
0.13373758
0.045044623
0.045044623
0.045044623
0.045044623
0.045044623
-0.028187057
-0.028187057
-0.028187057
-0.028187057
-0.028187057
-0.044709448
-0.044709448
-0.044709448
-0.044709448
-0.044709448
0.02774734
0.02774

0.025210122
0.025210122
0.025210122
0.025210122
0.025210122
-0.06319359
-0.06319359
-0.06319359
-0.06319359
-0.06319359
0.19274026
0.19274026
0.19274026
0.19274026
0.19274026
0.124579564
0.124579564
0.124579564
0.124579564
0.124579564
0.08659047
0.08659047
0.08659047
0.08659047
0.08659047
-0.09696121
-0.09696121
-0.09696121
-0.09696121
-0.09696121
-0.08377118
-0.08377118
-0.08377118
-0.08377118
-0.08377118
-0.07597294
-0.07597294
-0.07597294
-0.07597294
-0.07597294
0.024194326
0.024194326
0.024194326
0.024194326
0.024194326
0.019093513
0.019093513
0.019093513
0.019093513
0.019093513
0.030286757
0.030286757
0.030286757
0.030286757
0.030286757
0.15768054
0.15768054
0.15768054
0.15768054
0.15768054
-0.029096324
0.06279134
0.06279134
0.06279134
0.06279134
0.06279134
-0.07344544
-0.07344544
-0.07344544
-0.07344544
-0.07344544
0.045743532
0.045743532
0.045743532
0.045743532
0.045743532
-0.04742863
-0.04742863
-0.04742863
-0.04742863
-0.04742863
0.11197601
0.11197601
0.11197601
0.11197601
0.1

0.017284693
0.017284693
0.017284693
0.017284693
0.017284693
0.11098878
0.11098878
0.11098878
0.11098878
0.11098878
0.08439399
0.08439399
0.08439399
0.08439399
0.08439399
0.015662834
0.015662834
0.015662834
0.015662834
0.015662834
0.064794324
0.064794324
0.064794324
0.064794324
0.064794324
0.009380281
0.009380281
0.009380281
0.009380281
0.009380281
0.028983774
0.028983774
0.028983774
0.028983774
0.028983774
0.084911145
0.084911145
0.084911145
0.084911145
0.084911145
0.22065163
0.22065163
0.22065163
0.22065163
0.22065163
0.020842267
0.020842267
0.020842267
0.020842267
0.020842267
0.101038955
0.101038955
0.101038955
0.101038955
0.101038955
0.12567024
0.12567024
0.12567024
0.12567024
0.12567024
0.065303184
0.065303184
0.065303184
0.065303184
0.065303184
0.08406821
0.08406821
0.08406821
0.08406821
0.08406821
0.162746
0.162746
0.162746
0.162746
0.162746
0.09450052
0.09450052
0.09450052
0.09450052
0.09450052
0.09546086
0.09546086
0.09546086
0.09546086
0.09546086
0.02726337
0.02726337
0.027263

In [38]:
df_results['cosine'] = similarity
df_results['cosine'].describe()

count    1831.000000
mean        0.044432
std         0.079640
min        -0.152179
25%        -0.015455
50%         0.041070
75%         0.099817
max         0.240783
Name: cosine, dtype: float64